# Xây dựng ứng dụng tạo hình ảnh (OpenAI)

LLM không chỉ dùng để tạo văn bản. Bạn cũng có thể tạo hình ảnh từ mô tả bằng văn bản. Hình ảnh như một phương thức rất hữu ích trong lĩnh vực công nghệ y tế, kiến trúc, du lịch, phát triển trò chơi, tiếp thị, và nhiều lĩnh vực khác. Trong bài học này, chúng ta sẽ làm việc với các mô hình **GPT Image** ngày nay trên nền tảng OpenAI.

## Mục tiêu học tập

Đến cuối bài học này, bạn sẽ có thể:

- Giải thích về tạo hình ảnh là gì và nó hữu ích ở đâu.
- Hiểu về dòng mô hình `gpt-image` và cách nó khác với các mô hình DALL·E cũ.
- Xây dựng ứng dụng tạo hình ảnh và chỉnh sửa hình ảnh.

## Tạo hình ảnh là gì?

Các mô hình tạo hình ảnh tạo ra hình ảnh từ một câu lệnh văn bản. Các mô hình hiện đại như `gpt-image` học mối quan hệ giữa văn bản và hình ảnh trong quá trình huấn luyện, sau đó lần lượt biến đổi nhiễu ngẫu nhiên thành một hình ảnh phù hợp với câu lệnh của bạn.

Hai dòng mô hình hình ảnh nổi tiếng là:

- **`gpt-image` (OpenAI)** — thế hệ hiện tại được sử dụng trong bài học này. Nó hỗ trợ tạo hình từ văn bản và chỉnh sửa hình ảnh (vẽ đè theo vùng có mặt nạ).
- **Midjourney** — một mô hình bên thứ ba phổ biến với dịch vụ riêng và quy trình làm việc dựa trên Discord.

> Các mô hình hình ảnh OpenAI cũ — **DALL·E 2** và **DALL·E 3** — đã lỗi thời. DALL·E 3 không còn khả dụng cho triển khai mới, và các tính năng như `create_variation` chỉ có trong DALL·E 2. Hãy sử dụng các mô hình `gpt-image` cho ứng dụng mới.

> **Quan trọng:** các mô hình `gpt-image` trả về hình ảnh tạo ra dưới dạng **base64** (`b64_json`), không phải URL. Mã của bạn sẽ giải mã chuỗi base64 thành bytes và lưu lại — không có URL hình ảnh để tải xuống.


## Xây dựng ứng dụng tạo hình ảnh đầu tiên của bạn

Vậy để xây dựng một ứng dụng tạo hình ảnh cần những gì? Bạn cần các thư viện sau:

- **python-dotenv**, bạn được khuyến nghị mạnh mẽ sử dụng thư viện này để giữ bí mật trong tệp *.env* tách biệt khỏi mã nguồn.
- **openai**, thư viện này sẽ giúp bạn tương tác với API của OpenAI.
- **pillow**, để làm việc với hình ảnh trong Python.


1. Tạo một tệp *.env* với nội dung sau:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Thu thập các thư viện trên vào một tệp có tên *requirements.txt* như sau:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Tiếp theo, tạo môi trường ảo và cài đặt các thư viện:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Đối với Windows, sử dụng các lệnh sau để tạo và kích hoạt môi trường ảo của bạn:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Thêm đoạn mã sau vào tệp có tên *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Tạo đối tượng OpenAI (đọc OPENAI_API_KEY từ file .env của bạn)
    client = openai.OpenAI()


    try:
        # Tạo ảnh bằng cách sử dụng API tạo ảnh
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Nhập văn bản yêu cầu ở đây
            size='1024x1024',
            n=1
        )
        # Đặt thư mục lưu trữ ảnh
        image_dir = os.path.join(os.curdir, 'images')

        # Nếu thư mục không tồn tại, tạo nó
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Khởi tạo đường dẫn ảnh (lưu ý định dạng file nên là png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # các mô hình gpt-image trả về ảnh dưới dạng base64 (b64_json), không phải URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Hiển thị ảnh trong trình xem ảnh mặc định
        image = Image.open(image_path)
        image.show()

    # bắt ngoại lệ
    except openai.BadRequestError as err:
        print(err)

    ```

Hãy giải thích đoạn mã này:

- Đầu tiên, chúng ta nhập các thư viện cần thiết, bao gồm thư viện OpenAI, thư viện dotenv, mô-đun base64 và thư viện Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Sau đó, chúng ta tạo client, nó đọc khóa API từ file ``.env`` của bạn.

    ```python
    # Tạo đối tượng OpenAI
    client = openai.OpenAI()
    ```

- Tiếp theo, chúng ta tạo ảnh:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Nhập văn bản yêu cầu của bạn ở đây
        size='1024x1024',
        n=1
    )
    ```

    Các mô hình `gpt-image` trả về hình ảnh dưới dạng chuỗi **base64** trong `data[0].b64_json`. Chúng ta giải mã nó thành byte và ghi vào tệp — không có URL để tải xuống.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Cuối cùng, chúng ta mở ảnh và sử dụng trình xem ảnh tiêu chuẩn để hiển thị nó:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Thêm chi tiết về việc tạo ảnh

Hãy xem các tham số của `images.generate`:

- **model**, là mô hình ảnh được sử dụng (ví dụ `gpt-image-1`).
- **prompt**, là câu lệnh văn bản dùng để tạo ảnh. Ở đây là "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, là kích thước của ảnh được tạo (ví dụ `1024x1024`, `1536x1024`, `1024x1536`, hoặc `"auto"`).
- **n**, là số lượng ảnh được tạo. Ở đây chúng ta tạo một ảnh.

> Các mô hình ảnh không sử dụng tham số `temperature` — đó là tham số điều khiển cho sinh văn bản. Để có sự đa dạng, gọi API lần nữa; để giảm sự đa dạng, hãy làm cho câu lệnh của bạn cụ thể hơn.

## Các khả năng bổ sung của việc tạo ảnh

Bạn đã thấy cách tạo một bức ảnh với vài dòng Python. Các mô hình `gpt-image` cũng có thể **chỉnh sửa** một bức ảnh hiện có. Bằng cách cung cấp một ảnh, một **mask** tùy chọn (đánh dấu vùng cần thay đổi), cùng với câu lệnh, bạn có thể thay đổi một phần của bức ảnh — ví dụ, thêm mũ cho con thỏ của chúng ta.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# các chỉnh sửa cũng được trả về dưới dạng base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Ảnh gốc chỉ có con thỏ; ảnh cuối cùng có mũ trên thỏ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
